# TORGO Fine-tuning Record

This notebook tracks the final experiment flow for TinyVAD on TORGO:

1. Prepare TORGO data.
2. Run LOSO baseline.
3. Tune LOSO learning rate and re-evaluate.
4. Run mixed-data fine-tuning (TORGO + LibriParty) to reduce forgetting.
5. Evaluate the final trade-off between TORGO gain and LibriParty retention.

In [2]:
import os
os.chdir("/courses/CS6140.202630/students/cong.m")
!mkdir -p TORGO

# Download and extract
!wget -O - "https://www.cs.toronto.edu/~complingweb/data/TORGO/F.tar.bz2"  | tar -xjf - -C TORGO/
!wget -O - "https://www.cs.toronto.edu/~complingweb/data/TORGO/FC.tar.bz2" | tar -xjf - -C TORGO/
!wget -O - "https://www.cs.toronto.edu/~complingweb/data/TORGO/M.tar.bz2"  | tar -xjf - -C TORGO/
!wget -O - "https://www.cs.toronto.edu/~complingweb/data/TORGO/MC.tar.bz2" | tar -xjf - -C TORGO/

# Verify
!ls TORGO/

--2026-03-06 11:32:54--  https://www.cs.toronto.edu/~complingweb/data/TORGO/F.tar.bz2
Connecting to 10.99.0.130:3128... connected.
Proxy request sent, awaiting response... 200 OK
Length: 1139746435 (1.1G) [application/x-bzip2]
Saving to: ‘STDOUT’

-                   100%[===================>]   1.06G  2.18MB/s    in 4m 0s   

2026-03-06 11:36:54 (4.54 MB/s) - written to stdout [1139746435/1139746435]

--2026-03-06 11:36:54--  https://www.cs.toronto.edu/~complingweb/data/TORGO/FC.tar.bz2
Connecting to 10.99.0.130:3128... connected.
Proxy request sent, awaiting response... 200 OK
Length: 2532029976 (2.4G) [application/x-bzip2]
Saving to: ‘STDOUT’

-                   100%[===================>]   2.36G  1.97MB/s    in 8m 0s   

2026-03-06 11:44:54 (5.04 MB/s) - written to stdout [2532029976/2532029976]

--2026-03-06 11:44:55--  https://www.cs.toronto.edu/~complingweb/data/TORGO/M.tar.bz2
Connecting to 10.99.0.130:3128... connected.
Proxy request sent, awaiting response... 200 OK
Length: 

## LOSO Baseline Run

**LOSO (Leave-One-Speaker-Out)** means one speaker is held out for testing in each fold, and the model is trained on all remaining speakers.

This run establishes the speaker-independent baseline with the initial hyperparameters.

In [2]:
!python finetune_torgo_loso.py

Device: cuda
Loaded pre-trained weights from /courses/CS6140.202630/students/cong.m/results/TinyVAD/best_tinyvad.pt

Fold: test=F01  train=['F03', 'F04', 'M01', 'M02', 'M04', 'M05', 'MC01']
  Speakers ['F03', 'F04', 'M01', 'M02', 'M04', 'M05', 'MC01']: 2958 annotated files
    -> 3120 chunks, speech ratio: 27.14%
  Speakers ['F01']: 134 annotated files
    -> 137 chunks, speech ratio: 23.75%
  Epoch   1/20 | Loss: 0.3396 | F1: 0.8634 | Prec: 0.7989 | Rec: 0.9393
    --> Saved (F1=0.8634)
  Epoch   2/20 | Loss: 0.3072 | F1: 0.8724 | Prec: 0.8147 | Rec: 0.9390
    --> Saved (F1=0.8724)
  Epoch   3/20 | Loss: 0.2924 | F1: 0.8664 | Prec: 0.7990 | Rec: 0.9461
  Epoch   4/20 | Loss: 0.2815 | F1: 0.8641 | Prec: 0.7995 | Rec: 0.9399
  Epoch   5/20 | Loss: 0.2726 | F1: 0.8539 | Prec: 0.7847 | Rec: 0.9365
  Epoch   6/20 | Loss: 0.2639 | F1: 0.8442 | Prec: 0.7732 | Rec: 0.9296
  Epoch   7/20 | Loss: 0.2586 | F1: 0.8379 | Prec: 0.7652 | Rec: 0.9258
  Epoch   8/20 | Loss: 0.2541 | F1: 0.8327 | Prec

## Baseline Evaluation

Evaluate the baseline model after LOSO training.

Focus on:
- TORGO performance (adaptation quality)
- LibriParty retention (catastrophic forgetting)

In [2]:
!python eval_tinyvad.py

Loading TORGO (all annotated speakers)...
Loading LibriParty val set...
  1932 examples

[Baseline] Pre-trained TinyVAD ...
  LibriParty val : F1=0.9223  Prec=0.9168  Rec=0.9280
  TORGO (all)    : F1=0.7368  Prec=0.6344  Rec=0.8786

[Ceiling] CRDNN teacher ...
torchvision is not available - cannot save figures
  LibriParty val : F1=0.9463  Prec=0.9446  Rec=0.9481
  TORGO (all)    : F1=0.6320  Prec=0.4923  Rec=0.8825

[F01] Loading /courses/CS6140.202630/students/cong.m/results/TinyVAD/loso/tinyvad_loso_F01.pt
  TORGO  (F01): F1=0.8724  Prec=0.8147  Rec=0.9390  (gain vs pre-trained: +0.0813)
  LibriParty val  : F1=0.8563  Prec=0.9610  Rec=0.7722  (drop: +0.0660)

[F03] Loading /courses/CS6140.202630/students/cong.m/results/TinyVAD/loso/tinyvad_loso_F03.pt
  TORGO  (F03): F1=0.8780  Prec=0.8714  Rec=0.8846  (gain vs pre-trained: +0.0971)
  LibriParty val  : F1=0.8578  Prec=0.9602  Rec=0.7750  (drop: +0.0646)

[F04] Loading /courses/CS6140.202630/students/cong.m/results/TinyVAD/loso/tinyv

## LOSO Adjustment: Lower Learning Rate

Adjust LOSO learning rate from `1e-4` to `5e-5`, then rerun training and evaluate again.

Goal: improve optimization stability and reduce forgetting while keeping TORGO gains.

In [1]:
!python finetune_torgo_loso.py

Device: cuda
Loaded pre-trained weights from /courses/CS6140.202630/students/cong.m/results/TinyVAD/best_tinyvad.pt

Fold: test=F01  train=['F03', 'F04', 'M01', 'M02', 'M04', 'M05', 'MC01']
  Speakers ['F03', 'F04', 'M01', 'M02', 'M04', 'M05', 'MC01']: 2958 annotated files
    -> 3120 chunks, speech ratio: 27.14%
  Speakers ['F01']: 134 annotated files
    -> 137 chunks, speech ratio: 23.75%
  Epoch   1/20 | Loss: 0.3954 | F1: 0.8073 | Prec: 0.6915 | Rec: 0.9698
    --> Saved (F1=0.8073)
  Epoch   2/20 | Loss: 0.3711 | F1: 0.8193 | Prec: 0.7124 | Rec: 0.9640
    --> Saved (F1=0.8193)
  Epoch   3/20 | Loss: 0.3558 | F1: 0.8308 | Prec: 0.7332 | Rec: 0.9582
    --> Saved (F1=0.8308)
  Epoch   4/20 | Loss: 0.3429 | F1: 0.8389 | Prec: 0.7491 | Rec: 0.9532
    --> Saved (F1=0.8389)
  Epoch   5/20 | Loss: 0.3374 | F1: 0.8453 | Prec: 0.7626 | Rec: 0.9481
    --> Saved (F1=0.8453)
  Epoch   6/20 | Loss: 0.3302 | F1: 0.8505 | Prec: 0.7729 | Rec: 0.9453
    --> Saved (F1=0.8505)
  Epoch   7/20 | 

In [2]:
!python eval_tinyvad.py

Loading TORGO (all annotated speakers)...
Loading LibriParty val set...
  1932 examples

[Baseline] Pre-trained TinyVAD ...
  LibriParty val : F1=0.9223  Prec=0.9168  Rec=0.9280
  TORGO (all)    : F1=0.7368  Prec=0.6344  Rec=0.8786

[Ceiling] CRDNN teacher ...
torchvision is not available - cannot save figures
  LibriParty val : F1=0.9463  Prec=0.9446  Rec=0.9481
  TORGO (all)    : F1=0.6320  Prec=0.4923  Rec=0.8825

[F01] Loading /courses/CS6140.202630/students/cong.m/results/TinyVAD/loso_lr5e5/tinyvad_loso_F01.pt
  TORGO  (F01): F1=0.8726  Prec=0.8126  Rec=0.9421  (gain vs pre-trained: +0.0815)
  LibriParty val  : F1=0.8593  Prec=0.9606  Rec=0.7774  (drop: +0.0630)

[F03] Loading /courses/CS6140.202630/students/cong.m/results/TinyVAD/loso_lr5e5/tinyvad_loso_F03.pt
  TORGO  (F03): F1=0.8722  Prec=0.8620  Rec=0.8828  (gain vs pre-trained: +0.0913)
  LibriParty val  : F1=0.8604  Prec=0.9608  Rec=0.7790  (drop: +0.0619)

[F04] Loading /courses/CS6140.202630/students/cong.m/results/TinyVA

## Mixed-data Fine-tuning (Anti-forgetting)

Run `finetune_mixed.py`, which mixes TORGO data with LibriParty data during training.

Purpose:
- Keep adaptation to dysarthric speech (TORGO)
- Preserve general VAD ability from LibriParty
- Reduce catastrophic forgetting

## Mixed Run Evaluation Setup

After mixed-data fine-tuning, run evaluation to measure the final TORGO-vs-LibriParty trade-off.

In [1]:
!python finetune_mixed.py

Device: cuda
Loaded pre-trained weights from /courses/CS6140.202630/students/cong.m/results/TinyVAD/best_tinyvad.pt

Loading LibriParty train set for replay...
  LibriParty replay: 11180 examples from /courses/CS6140.202630/students/cong.m/results/VAD/save/train.json

Fold: test=F01  train=['F03', 'F04', 'M01', 'M02', 'M04', 'M05', 'MC01']
  Speakers ['F03', 'F04', 'M01', 'M02', 'M04', 'M05', 'MC01']: 2958 annotated files
    -> 3120 chunks, speech ratio: 27.14%
  Speakers ['F01']: 134 annotated files
    -> 137 chunks, speech ratio: 23.75%
  Epoch   1/20 | Loss: 0.4438 | F1: 0.8523 | Prec: 0.7682 | Rec: 0.9572
    --> Saved (F1=0.8523)
  Epoch   2/20 | Loss: 0.4036 | F1: 0.8594 | Prec: 0.7802 | Rec: 0.9565
    --> Saved (F1=0.8594)
  Epoch   3/20 | Loss: 0.3833 | F1: 0.8516 | Prec: 0.7665 | Rec: 0.9579
  Epoch   4/20 | Loss: 0.3694 | F1: 0.8478 | Prec: 0.7629 | Rec: 0.9539
  Epoch   5/20 | Loss: 0.3574 | F1: 0.8434 | Prec: 0.7598 | Rec: 0.9477
  Epoch   6/20 | Loss: 0.3428 | F1: 0.836

In [2]:
!python eval_tinyvad.py

Loading TORGO (all annotated speakers)...
Loading LibriParty val set...
  1932 examples

[Baseline] Pre-trained TinyVAD ...
  LibriParty val : F1=0.9223  Prec=0.9168  Rec=0.9280
  TORGO (all)    : F1=0.7368  Prec=0.6344  Rec=0.8786

[Ceiling] CRDNN teacher ...
torchvision is not available - cannot save figures
  LibriParty val : F1=0.9463  Prec=0.9446  Rec=0.9481
  TORGO (all)    : F1=0.6320  Prec=0.4923  Rec=0.8825

[F01] Loading /courses/CS6140.202630/students/cong.m/results/TinyVAD/loso_mixed/tinyvad_loso_F01.pt
  TORGO  (F01): F1=0.8594  Prec=0.7802  Rec=0.9565  (gain vs pre-trained: +0.0683)
  LibriParty val  : F1=0.9171  Prec=0.9241  Rec=0.9102  (drop: +0.0052)

[F03] Loading /courses/CS6140.202630/students/cong.m/results/TinyVAD/loso_mixed/tinyvad_loso_F03.pt
  TORGO  (F03): F1=0.8709  Prec=0.8361  Rec=0.9087  (gain vs pre-trained: +0.0900)
  LibriParty val  : F1=0.9147  Prec=0.9034  Rec=0.9263  (drop: +0.0076)

[F04] Loading /courses/CS6140.202630/students/cong.m/results/TinyVA